#  Fine-Tuning Llama-3.1-8B-Instruct for Document Intelligence RAG

This notebook provides a complete end-to-end pipeline to:
1. **Install dependencies** (Unsloth, PEFT, Transformers, vLLM).
2. **Load base model** `unsloth/meta-llama-3.1-8b-instruct-bnb-4bit`.
3. **Format custom dataset** for QA and structured JSON extraction.
4. **Train using QLoRA** (fast 2x memory-efficient fine-tuning).
5. **Merge LoRA weights into 16-bit base model**.
6. **Export / Push merged model** to Hugging Face Hub for **vLLM serving**.

### Step 1: Install Unsloth & Requirements

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install vllm datasets huggingface_hub

### Step 2: Load Pre-Quantized Base Model (Llama-3.1-8B-Instruct)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096  # Max context window for document chunks
dtype = None  # None for auto detection (Float16 for Tesla T4, Bfloat16 for A100/L4)
load_in_4bit = True  # Use 4-bit quantization to fit within Google Colab free GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Step 3: Attach QLoRA Adapters to Model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### Step 4: Prepare Document QA & JSON Extraction Training Dataset

In [ ]:
import json
from datasets import Dataset

# Load 100 fine-tuning dataset samples generated for Document Intelligence RAG
with open('rag_finetune_100_samples.json', 'r') as f:
    training_samples = json.load(f)

def format_prompts(example):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    return { 'text': text }

dataset = Dataset.from_list(training_samples)
dataset = dataset.map(format_prompts)
print(f"✅ Successfully loaded and tokenized {len(dataset)} training samples!")

### Step 5: Execute Fine-Tuning Training Loop

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()


### Step 6: Merge LoRA Adapter with Base Model & Save for vLLM Serving

In [ ]:
# Merge weights into 16-bit precision directory for vLLM serving
merged_output_dir = "finetuned_llama3_8b_merged"
model.save_pretrained_merged(merged_output_dir, tokenizer, save_method = "merged_16bit")
print(f"✅ Merged fine-tuned model saved successfully to: {merged_output_dir}")

# Optional: Push merged model directly to Hugging Face Hub for easy vLLM loading
# model.push_to_hub_merged("your-hf-username/finetuned-llama3-8b-rag", tokenizer, save_method = "merged_16bit", token = "hf_...")